In [2]:
"""
Примеры кода "Управление промптами"

Этот модуль демонстрирует:
1. Работа с сообщениями (SystemMessage, HumanMessage, AIMessage)
2. ChatPromptTemplate и его методы
3. MessagesPlaceholder для динамической истории
4. Few-shot prompting
5. Partial formatting
6. Композиция промптов
7. Практические паттерны
"""

import os
from typing import List, Dict, Any
from datetime import datetime
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

# ============================================================================
# ЧАСТЬ 1: АНАТОМИЯ СООБЩЕНИЙ
# ============================================================================

In [4]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    BaseMessage,
)
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI


In [6]:
"""
Демонстрация типов сообщений в LangChain.
"""
print("=" * 60)
print("ПРИМЕР 1: Типы сообщений")
print("=" * 60)

# SystemMessage — инструкции для модели
system = SystemMessage(content="Ты — опытный Python-разработчик.")
print(f"\n1.1. SystemMessage:")
print(f"  type: {system.type}")
print(f"  content: {system.content}")

# HumanMessage — сообщение пользователя
human = HumanMessage(content="Как создать список в Python?")
print(f"\n1.2. HumanMessage:")
print(f"  type: {human.type}")
print(f"  content: {human.content}")

# AIMessage — ответ ассистента
ai = AIMessage(content="Список создаётся так: my_list = [1, 2, 3]")
print(f"\n1.3. AIMessage:")
print(f"  type: {ai.type}")
print(f"  content: {ai.content}")

# Использование в диалоге
print("\n1.4. Диалог из сообщений:")
messages = [system, human, ai, HumanMessage(content="А словарь?")]

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)
response = model.invoke(messages)
print(f"  Ответ модели: {response.content}")

ПРИМЕР 1: Типы сообщений

1.1. SystemMessage:
  type: system
  content: Ты — опытный Python-разработчик.

1.2. HumanMessage:
  type: human
  content: Как создать список в Python?

1.3. AIMessage:
  type: ai
  content: Список создаётся так: my_list = [1, 2, 3]

1.4. Диалог из сообщений:
  Ответ модели: Словарь в Python создаётся с помощью фигурных скобок `{}` или с помощью функции `dict()`. Вот несколько примеров:

1. С использованием фигурных скобок:
```python
my_dict = {'ключ1': 'значение1', 'ключ2': 'значение2'}
```

2. С использованием функции `dict()`:
```python
my_dict = dict(ключ1='значение1', ключ2='значение2')
```

В обоих случаях вы получите словарь, где ключи связаны со значениями.


In [7]:
"""
Дополнительные атрибуты сообщений.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 2: Метаданные сообщений")
print("=" * 60)

# Сообщение с дополнительными атрибутами
message = HumanMessage(
    content="Привет!",
    additional_kwargs={"source": "web_chat"},
    id="msg_001",
    name="Алексей"
)

print("\n2.1. Атрибуты сообщения:")
print(f"  content: {message.content}")
print(f"  type: {message.type}")
print(f"  id: {message.id}")
print(f"  name: {message.name}")
print(f"  additional_kwargs: {message.additional_kwargs}")



ПРИМЕР 2: Метаданные сообщений

2.1. Атрибуты сообщения:
  content: Привет!
  type: human
  id: msg_001
  name: Алексей
  additional_kwargs: {'source': 'web_chat'}


In [8]:
"""
Мультимодальные сообщения (текст + изображения).
"""
print("\n" + "=" * 60)
print("ПРИМЕР 3: Мультимодальные сообщения")
print("=" * 60)

# Сообщение с текстом и изображением (URL)
multimodal_message = HumanMessage(
    content=[
        {"type": "text", "text": "Что изображено на этой картинке?"},
        {
            "type": "image_url",
            "image_url": {"url": "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c3/Python-logo-notext.svg/200px-Python-logo-notext.svg.png"}
        }
    ]
)

print("\n3.1. Структура мультимодального сообщения:")
print(f"  Тип content: {type(multimodal_message.content)}")
print(f"  Количество элементов: {len(multimodal_message.content)}")
for i, item in enumerate(multimodal_message.content):
    print(f"  Элемент {i}: type={item['type']}")

# Вызов с мультимодальным сообщением
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
response = model.invoke([multimodal_message])
print(f"\n3.2. Ответ модели: {response.content}")



ПРИМЕР 3: Мультимодальные сообщения

3.1. Структура мультимодального сообщения:
  Тип content: <class 'list'>
  Количество элементов: 2
  Элемент 0: type=text
  Элемент 1: type=image_url

3.2. Ответ модели: На изображении представлен логотип языка программирования Python. Он состоит из двух переплетающихся фигур, напоминающих змей, выполненных в синих и желтых цветах.


# ============================================================================
# ЧАСТЬ 2: CHATPROMPTTEMPLATE
# ============================================================================

In [9]:
"""
Базовое создание ChatPromptTemplate.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 4: Базовый ChatPromptTemplate")
print("=" * 60)

# Способ 1: from_messages с кортежами
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "Ты — эксперт в области {domain}."),
    ("human", "{question}")
])

print("\n4.1. Создание через from_messages (кортежи):")
print(f"  Переменные: {prompt1.input_variables}")

# Способ 2: from_messages с классами
prompt2 = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Ты — эксперт в области {domain}."),
    HumanMessagePromptTemplate.from_template("{question}")
])

print("\n4.2. Создание через классы сообщений:")
print(f"  Переменные: {prompt2.input_variables}")

# Форматирование
formatted = prompt1.invoke({
    "domain": "машинного обучения",
    "question": "Что такое градиентный спуск?"
})

print("\n4.3. Отформатированные сообщения:")
for msg in formatted.messages:
    print(f"  [{msg.__class__.__name__}]: {msg.content}")



ПРИМЕР 4: Базовый ChatPromptTemplate

4.1. Создание через from_messages (кортежи):
  Переменные: ['domain', 'question']

4.2. Создание через классы сообщений:
  Переменные: ['domain', 'question']

4.3. Отформатированные сообщения:
  [SystemMessage]: Ты — эксперт в области машинного обучения.
  [HumanMessage]: Что такое градиентный спуск?


In [11]:
"""
Создание простого промпта из шаблона.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 5: ChatPromptTemplate.from_template")
print("=" * 60)

# Простой шаблон с одним сообщением
simple_prompt = ChatPromptTemplate.from_template(
    "Переведи на {language}: {text}"
)

print("\n5.1. Простой шаблон:")
print(f"  Переменные: {simple_prompt.input_variables}")

formatted = simple_prompt.invoke({"language": "английский", "text": "Привет мир"})
print(f"  Сообщение: {formatted.messages[0].content}")

# Использование в цепочке
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
chain = simple_prompt | model | StrOutputParser()

result = chain.invoke({"language": "французский", "text": "Добрый день"})
print(f"\n5.2. Результат цепочки: {result}")



ПРИМЕР 5: ChatPromptTemplate.from_template

5.1. Простой шаблон:
  Переменные: ['language', 'text']
  Сообщение: Переведи на английский: Привет мир

5.2. Результат цепочки: Добрый день на французском будет "Bonjour".


In [12]:
"""
Инспекция схемы промпта.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 6: Схема промпта")
print("=" * 60)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Контекст: {context}"),
    ("human", "Вопрос: {question}\nФормат ответа: {format}")
])

print("\n6.1. Схема входных данных:")
schema = prompt.input_schema.schema()
print(f"  properties: {list(schema['properties'].keys())}")
print(f"  required: {schema.get('required', [])}")

print("\n6.2. Все переменные шаблона:")
print(f"  {prompt.input_variables}")


ПРИМЕР 6: Схема промпта

6.1. Схема входных данных:
  properties: ['context', 'format', 'question']
  required: ['context', 'format', 'question']

6.2. Все переменные шаблона:
  ['context', 'format', 'question']


/var/folders/d_/k331795x38q58drfdtws24lh0000gn/T/ipykernel_29347/2684947876.py:14: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  schema = prompt.input_schema.schema()


# ============================================================================
# ЧАСТЬ 3: MESSAGESPLACEHOLDER
# ============================================================================


In [14]:
"""
MessagesPlaceholder для динамического списка сообщений.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 7: MessagesPlaceholder")
print("=" * 60)

# Промпт с плейсхолдером для истории
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — дружелюбный ассистент."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

print("\n7.1. Переменные промпта:")
print(f"  {prompt.input_variables}")

# История диалога
history = [
    HumanMessage(content="Меня зовут Алексей"),
    AIMessage(content="Приятно познакомиться, Алексей!"),
    HumanMessage(content="Я изучаю Python"),
    AIMessage(content="Отличный выбор! Python — замечательный язык.")
]

# Форматирование с историей
formatted = prompt.invoke({
    "history": history,
    "input": "Как меня зовут?"
})

print("\n7.2. Все сообщения после форматирования:")
for i, msg in enumerate(formatted.messages):
    print(f"  {i}. [{msg.__class__.__name__}]: {msg.content[:50]}...")

# Вызов модели
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
response = model.invoke(formatted)
print(f"\n7.3. Ответ модели: {response.content}")


ПРИМЕР 7: MessagesPlaceholder

7.1. Переменные промпта:
  ['history', 'input']

7.2. Все сообщения после форматирования:
  0. [SystemMessage]: Ты — дружелюбный ассистент....
  1. [HumanMessage]: Меня зовут Алексей...
  2. [AIMessage]: Приятно познакомиться, Алексей!...
  3. [HumanMessage]: Я изучаю Python...
  4. [AIMessage]: Отличный выбор! Python — замечательный язык....
  5. [HumanMessage]: Как меня зовут?...

7.3. Ответ модели: Вас зовут Алексей.


In [16]:
"""
Опциональный MessagesPlaceholder.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 8: Optional MessagesPlaceholder")
print("=" * 60)

# Промпт с опциональной историей
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — ассистент."),
    MessagesPlaceholder(variable_name="history", optional=True),
    ("human", "{input}")
])

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
chain = prompt | model | StrOutputParser()

# Вызов БЕЗ истории
print("\n8.1. Вызов без истории:")
result1 = chain.invoke({"input": "Привет!"})
print(f"  Ответ: {result1}")

# Вызов С историей
print("\n8.2. Вызов с историей:")
result2 = chain.invoke({
    "input": "А что мы обсуждали?",
    "history": [
        HumanMessage(content="Расскажи о Python"),
        AIMessage(content="Python — это язык программирования...")
    ]
})
print(f"  Ответ: {result2[:80]}...")


ПРИМЕР 8: Optional MessagesPlaceholder

8.1. Вызов без истории:
  Ответ: Привет! Как я могу помочь тебе сегодня?

8.2. Вызов с историей:
  Ответ: Мы еще не обсуждали ничего конкретного. Вы задали вопрос о Python, и я начал рас...


In [17]:
"""
Обрезка истории сообщений.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 9: trim_messages")
print("=" * 60)

from langchain_core.messages import trim_messages

# Длинная история
long_history = []
for i in range(10):
    long_history.extend([
        HumanMessage(content=f"Вопрос номер {i+1}: Расскажи что-нибудь интересное"),
        AIMessage(content=f"Ответ {i+1}: Вот интересный факт номер {i+1}...")
    ])

print(f"\n9.1. Исходная история: {len(long_history)} сообщений")

# Обрезка по токенам
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
)

trimmed = trim_messages(
    long_history,
    max_tokens=200,
    strategy="last",  # Оставляем последние
    token_counter=model,
    include_system=True
)

print(f"\n9.2. После обрезки: {len(trimmed)} сообщений")
for msg in trimmed:
    print(f"  [{msg.__class__.__name__}]: {msg.content[:40]}...")



ПРИМЕР 9: trim_messages

9.1. Исходная история: 20 сообщений

9.2. После обрезки: 11 сообщений
  [AIMessage]: Ответ 5: Вот интересный факт номер 5......
  [HumanMessage]: Вопрос номер 6: Расскажи что-нибудь инте...
  [AIMessage]: Ответ 6: Вот интересный факт номер 6......
  [HumanMessage]: Вопрос номер 7: Расскажи что-нибудь инте...
  [AIMessage]: Ответ 7: Вот интересный факт номер 7......
  [HumanMessage]: Вопрос номер 8: Расскажи что-нибудь инте...
  [AIMessage]: Ответ 8: Вот интересный факт номер 8......
  [HumanMessage]: Вопрос номер 9: Расскажи что-нибудь инте...
  [AIMessage]: Ответ 9: Вот интересный факт номер 9......
  [HumanMessage]: Вопрос номер 10: Расскажи что-нибудь инт...
  [AIMessage]: Ответ 10: Вот интересный факт номер 10.....


# ============================================================================
# ЧАСТЬ 4: FEW-SHOT PROMPTING
# ============================================================================


In [19]:
"""
Ручное создание few-shot промпта.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 10: Few-shot (ручной способ)")
print("=" * 60)

examples = [
    {"input": "Я люблю этот продукт!", "output": "позитивный"},
    {"input": "Ужасное качество", "output": "негативный"},
    {"input": "Нормально, ничего особенного", "output": "нейтральный"}
]

# Разворачиваем примеры в сообщения
example_messages = []
for ex in examples:
    example_messages.extend([
        ("human", ex["input"]),
        ("ai", ex["output"])
    ])

prompt = ChatPromptTemplate.from_messages([
    ("system", "Определи тональность текста. Ответь одним словом: позитивный, негативный или нейтральный."),
    *example_messages,
    ("human", "{text}")
])

print("\n10.1. Структура промпта:")
print(f"  Всего сообщений: {len(prompt.messages)}")

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
chain = prompt | model | StrOutputParser()

test_texts = [
    "Замечательный сервис!",
    "Больше никогда не закажу",
    "Доставка была в срок"
]

print("\n10.2. Классификация:")
for text in test_texts:
    result = chain.invoke({"text": text})
    print(f"  '{text}' -> {result}")


ПРИМЕР 10: Few-shot (ручной способ)

10.1. Структура промпта:
  Всего сообщений: 8

10.2. Классификация:
  'Замечательный сервис!' -> позитивный
  'Больше никогда не закажу' -> негативный
  'Доставка была в срок' -> позитивный


In [21]:
"""
FewShotChatMessagePromptTemplate для управления примерами.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 11: FewShotChatMessagePromptTemplate")
print("=" * 60)


from langchain_core.prompts import FewShotChatMessagePromptTemplate


examples = [
    {"input": "2 + 2", "output": "4"},
    {"input": "5 * 3", "output": "15"},
    {"input": "10 / 2", "output": "5"}
]

# Шаблон для одного примера
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# Few-shot шаблон
few_shot = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt
)

# Финальный промпт
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — калькулятор. Вычисляй выражения."),
    few_shot,
    ("human", "{input}")
])

print("\n11.1. Переменные промпта:")
print(f"  {final_prompt.input_variables}")

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
chain = final_prompt | model | StrOutputParser()

result = chain.invoke({"input": "7 + 8"})
print(f"\n11.2. Результат: 7 + 8 = {result}")


ПРИМЕР 11: FewShotChatMessagePromptTemplate

11.1. Переменные промпта:
  ['input']

11.2. Результат: 7 + 8 = 15


In [7]:
"""
Динамический выбор примеров по релевантности.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 12: Динамический выбор примеров")
print("=" * 60)


from langchain_core.prompts import FewShotChatMessagePromptTemplate
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


examples = [
    {"input": "Как создать список?", "output": "my_list = [1, 2, 3]", "topic": "python"},
    {"input": "Как создать словарь?", "output": "my_dict = {'key': 'value'}", "topic": "python"},
    {"input": "Как сделать цикл?", "output": "for i in range(10): print(i)", "topic": "python"},
    {"input": "Как установить пакет?", "output": "pip install package_name", "topic": "pip"},
    {"input": "Как создать виртуальное окружение?", "output": "python -m venv myenv", "topic": "venv"},
]

# Создаём selector на основе семантической близости
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    OpenAIEmbeddings(
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url=os.getenv("OPENROUTER_BASE_URL"),
        model="openai/text-embedding-3-small"
    ),
    FAISS,
    k=2  # Выбрать 2 самых похожих примера
)

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — Python-эксперт. Отвечай кодом."),
    few_shot,
    ("human", "{input}")
])

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
chain = final_prompt | model | StrOutputParser()

print("\n12.1. Запрос про структуры данных:")
result = chain.invoke({"input": "Как создать множество?"})
print(f"  Ответ: {result}")

print("\n12.2. Запрос про окружение:")
result = chain.invoke({"input": "Как активировать окружение?"})
print(f"  Ответ: {result}")



ПРИМЕР 12: Динамический выбор примеров

12.1. Запрос про структуры данных:
  Ответ: my_set = {1, 2, 3}

12.2. Запрос про окружение:
  Ответ: Для Windows:
```bash
myenv\Scripts\activate
```

Для macOS и Linux:
```bash
source myenv/bin/activate
```


# ============================================================================
# ЧАСТЬ 5: PARTIAL FORMATTING
# ============================================================================

In [27]:
"""
Частичное заполнение с фиксированными значениями.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 13: Partial с фиксированными значениями")
print("=" * 60)

base_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — ассистент компании {company}. Отвечай в стиле: {tone}"),
    ("human", "{question}")
])

print("\n13.1. Исходные переменные:")
print(f"  {base_prompt.input_variables}")

# Частично заполняем известные значения
company_prompt = base_prompt.partial(company="TechCorp", tone="дружелюбный")

print("\n13.2. После partial:")
print(f"  {company_prompt.input_variables}")

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
chain = company_prompt | model | StrOutputParser()

result = chain.invoke({"question": "Какие у вас часы работы?"})
print(f"\n13.3. Ответ: {result}")



ПРИМЕР 13: Partial с фиксированными значениями

13.1. Исходные переменные:
  ['company', 'question', 'tone']

13.2. После partial:
  ['question']

13.3. Ответ: Привет! Мы работаем с понедельника по пятницу с 9:00 до 18:00. Если у тебя есть вопросы или нужна помощь, не стесняйся обращаться! 😊


In [28]:
"""
Частичное заполнение с функциями.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 14: Partial с функциями")
print("=" * 60)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Текущая дата: {date}. Текущее время: {time}."),
    ("human", "{question}")
])

def get_date():
    return datetime.now().strftime("%d.%m.%Y")

def get_time():
    return datetime.now().strftime("%H:%M")

# Функции вызываются при каждом invoke
dynamic_prompt = prompt.partial(date=get_date, time=get_time)

print("\n14.1. Первый вызов:")
formatted1 = dynamic_prompt.invoke({"question": "Сколько сейчас времени?"})
print(f"  System: {formatted1.messages[0].content}")


import time


time.sleep(1)

print("\n14.2. Второй вызов (через 1 сек):")
formatted2 = dynamic_prompt.invoke({"question": "Сколько сейчас времени?"})
print(f"  System: {formatted2.messages[0].content}")



ПРИМЕР 14: Partial с функциями

14.1. Первый вызов:
  System: Текущая дата: 09.02.2026. Текущее время: 04:58.

14.2. Второй вызов (через 1 сек):
  System: Текущая дата: 09.02.2026. Текущее время: 04:58.


# ============================================================================
# ЧАСТЬ 6: КОМПОЗИЦИЯ ПРОМПТОВ
# ============================================================================

In [29]:
"""
Объединение промптов через оператор +.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 15: Конкатенация промптов")
print("=" * 60)

# Базовые компоненты
system_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — {role}.")
])

context_prompt = ChatPromptTemplate.from_messages([
    ("system", "Контекст: {context}")
])

user_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

# Объединение
combined = system_prompt + context_prompt + user_prompt

print("\n15.1. Объединённый промпт:")
print(f"  Переменные: {combined.input_variables}")
print(f"  Количество сообщений: {len(combined.messages)}")

formatted = combined.invoke({
    "role": "юрист",
    "context": "Трудовой кодекс РФ",
    "question": "Сколько дней отпуска положено?"
})

print("\n15.2. Отформатированные сообщения:")
for msg in formatted.messages:
    print(f"  [{msg.__class__.__name__}]: {msg.content}")



ПРИМЕР 15: Конкатенация промптов

15.1. Объединённый промпт:
  Переменные: ['context', 'question', 'role']
  Количество сообщений: 3

15.2. Отформатированные сообщения:
  [SystemMessage]: Ты — юрист.
  [SystemMessage]: Контекст: Трудовой кодекс РФ
  [HumanMessage]: Сколько дней отпуска положено?


In [40]:
"""
PipelinePromptTemplate для иерархических промптов.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 16: PipelinePromptTemplate")
print("=" * 60)


from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.pipeline import PipelinePromptTemplate


# Компоненты
persona = PromptTemplate.from_template("Ты — {role}.")
style = PromptTemplate.from_template("Отвечай {style_description}.")
task = PromptTemplate.from_template("Твоя задача: {task_description}")

# Финальный шаблон
final = PromptTemplate.from_template(
    """{persona}
{style}
{task}

Вопрос: {question}"""
)

pipeline = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=[
        ("persona", persona),
        ("style", style),
        ("task", task)
    ]
)

print("\n16.1. Переменные pipeline:")
print(f"  {pipeline.input_variables}")

result = pipeline.invoke({
    "role": "учитель математики",
    "style_description": "простым языком, с примерами",
    "task_description": "объяснять сложные концепции",
    "question": "Что такое производная?"
})

print("\n16.2. Результат:")
print(result["text"])


ПРИМЕР 16: PipelinePromptTemplate


ModuleNotFoundError: No module named 'langchain_core.prompts.pipeline'

In [ ]:
"""
В версиях LangChain 1.x (2026 год) иерархические промпты больше не требуют 
сложного класса  PipelinePromptTemplate . Вместо него используется композиция 
объектов через оператор сложения  +  или прямое форматирование. Это работает 
быстрее и полностью поддерживает типизацию LCEL.

Теперь объекты  PromptTemplate  можно просто складывать друг с другом. 
Это автоматически объединяет их переменные.
"""

from langchain_core.prompts import PromptTemplate

# 1. Определяем компоненты как отдельные шаблоны
persona = PromptTemplate.from_template("Ты — {role}.\n")
style = PromptTemplate.from_template("Отвечай {style_description}.\n")
task = PromptTemplate.from_template("Твоя задача: {task_description}\n")
question = PromptTemplate.from_template("\nВопрос: {question}")

# 2. Просто складываем их (это заменяет PipelinePromptTemplate)
pipeline = persona + style + task + question

print("\n" + "=" * 60)
print("ОБНОВЛЕННЫЙ ПРИМЕР: Prompt Composition")
print("=" * 60)

print("\nВходные переменные:")
print(f"  {pipeline.input_variables}")

# 3. Вызываем через invoke
# В 1.x invoke возвращает StringPromptValue (используем .to_string() или просто print)
result = pipeline.invoke({
    "role": "учитель математики",
    "style_description": "простым языком, с примерами",
    "task_description": "объяснять сложные концепции",
    "question": "Что такое производная?"
})

print("\nРезультат:")
print(result.to_string())


ОБНОВЛЕННЫЙ ПРИМЕР: Prompt Composition

Входные переменные:
  ['question', 'role', 'style_description', 'task_description']

Результат:
Ты — учитель математики.
Отвечай простым языком, с примерами.
Твоя задача: объяснять сложные концепции

Вопрос: Что такое производная?


# ============================================================================
# ЧАСТЬ 7: ПРАКТИЧЕСКИЕ ПАТТЕРНЫ
# ============================================================================

In [42]:
"""
Промпт для RAG (Retrieval-Augmented Generation).
"""
print("\n" + "=" * 60)
print("ПРИМЕР 17: RAG промпт")
print("=" * 60)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты — ассистент, отвечающий на вопросы на основе контекста.

Правила:
- Используй ТОЛЬКО информацию из контекста
- Если ответа нет в контексте, скажи "Информация не найдена"
- Цитируй источники"""),
    ("human", """Контекст:
---
{context}
---

Вопрос: {question}""")
])

# Имитация контекста от retriever'а
context = """
Документ 1: Python создан Гвидо ван Россумом в 1991 году.
Документ 2: Python назван в честь комедийной группы Monty Python.
Документ 3: Python — интерпретируемый язык с динамической типизацией.
"""

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
chain = rag_prompt | model | StrOutputParser()

print("\n17.1. Вопрос с ответом в контексте:")
result = chain.invoke({"context": context, "question": "Кто создал Python?"})
print(f"  Ответ: {result}")

print("\n17.2. Вопрос без ответа в контексте:")
result = chain.invoke({"context": context, "question": "Какая последняя версия Python?"})
print(f"  Ответ: {result}")



ПРИМЕР 17: RAG промпт

17.1. Вопрос с ответом в контексте:
  Ответ: Python создан Гвидо ван Россумом в 1991 году (Документ 1).

17.2. Вопрос без ответа в контексте:
  Ответ: Информация не найдена.


In [33]:
"""
Промпт для классификации.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 18: Промпт классификации")
print("=" * 60)

classification_prompt = ChatPromptTemplate.from_messages([
    ("system", """Классифицируй обращение клиента.

Категории:
- BILLING: вопросы об оплате, счетах, тарифах
- TECHNICAL: технические проблемы, баги
- SALES: интерес к покупке, скидки
- COMPLAINT: жалобы, недовольство
- OTHER: не относится к другим категориям

Ответь ТОЛЬКО названием категории."""),
    ("human", "{message}")
])

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
chain = classification_prompt | model | StrOutputParser()

messages = [
    "Не могу оплатить подписку, карта не проходит",
    "Приложение вылетает при загрузке",
    "Какие скидки на годовую подписку?",
    "Это уже третий раз когда вы опаздываете с доставкой!",
    "Когда следующий вебинар?"
]

print("\n18.1. Классификация обращений:")
for msg in messages:
    category = chain.invoke({"message": msg})
    print(f"  '{msg[:40]}...' -> {category}")



ПРИМЕР 18: Промпт классификации

18.1. Классификация обращений:
  'Не могу оплатить подписку, карта не прох...' -> BILLING
  'Приложение вылетает при загрузке...' -> TECHNICAL
  'Какие скидки на годовую подписку?...' -> SALES
  'Это уже третий раз когда вы опаздываете ...' -> COMPLAINT
  'Когда следующий вебинар?...' -> OTHER


In [34]:
"""
Промпт для агента с инструментами.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 19: Промпт для агента")
print("=" * 60)

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты — AI-ассистент с доступом к инструментам.

Доступные инструменты:
{tools}

Чтобы использовать инструмент, ответь в формате:
Action: название_инструмента
Action Input: входные данные

Чтобы дать финальный ответ:
Final Answer: твой ответ"""),
    MessagesPlaceholder(variable_name="agent_scratchpad", optional=True),
    ("human", "{input}")
])

tools_description = """
1. search: Поиск информации в интернете
Input: поисковый запрос
2. calculator: Вычисление математических выражений
Input: математическое выражение
3. weather: Получение погоды
Input: название города
"""

print("\n19.1. Форматирование промпта агента:")
formatted = agent_prompt.invoke({
    "tools": tools_description,
    "input": "Какая погода в Москве?"
})

for msg in formatted.messages:
    print(f"\n  [{msg.__class__.__name__}]:")
    print(f"  {msg.content[:200]}...")



ПРИМЕР 19: Промпт для агента

19.1. Форматирование промпта агента:

  [SystemMessage]:
  Ты — AI-ассистент с доступом к инструментам.

Доступные инструменты:

1. search: Поиск информации в интернете
Input: поисковый запрос
2. calculator: Вычисление математических выражений
Input: математи...

  [HumanMessage]:
  Какая погода в Москве?...


In [35]:
"""
Промпт для Chain of Thought рассуждений.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 20: Chain of Thought промпт")
print("=" * 60)

cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """Решай задачи пошагово.

Формат ответа:
Шаг 1: [описание шага]
Шаг 2: [описание шага]
...
Ответ: [финальный ответ]"""),
    ("human", "{problem}")
])

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0
)
chain = cot_prompt | model | StrOutputParser()

problem = "В корзине 5 яблок. Маша взяла 2 яблока, потом положила обратно 1. Сколько яблок в корзине?"

print(f"\n20.1. Задача: {problem}")
result = chain.invoke({"problem": problem})
print(f"\n20.2. Решение:\n{result}")



ПРИМЕР 20: Chain of Thought промпт

20.1. Задача: В корзине 5 яблок. Маша взяла 2 яблока, потом положила обратно 1. Сколько яблок в корзине?

20.2. Решение:
Шаг 1: В корзине изначально 5 яблок.  
Шаг 2: Маша взяла 2 яблока, значит в корзине осталось 5 - 2 = 3 яблока.  
Шаг 3: Маша положила обратно 1 яблоко, значит в корзине стало 3 + 1 = 4 яблока.  

Ответ: 4 яблока.
